# LULC

This notebook does 3 things:
1. Generate training data for LULC classification. It uses 3 existing LULC products.
2. Filter training data per class using kmeans clustering to exclude outliers.
3. Train a random forest model, and then predict LULC.

### Steps: 
1. Training data:
For each site extract training data.
  -> Find product agreement. mask to gadm 100m buffer.
  -> Add geomad, indices, elevation etc.
  -> Write csv per site of training data.
do this in a notebook. don't commit training data.

1a. Filter outliers per class.

2. Train model:
Try one model for all sites. (append all CSVs)
export model dump (python pickle?)
in future we may need to make different models for different regions and year ranges.
train the model using the geomad of the year of the input products.

3. Predict:
per grid tile:
  per year:
    load geomad/indices/elevation etc.
    make using get_gadm (buffered 100m)
    predict
This is as a command. 

### Load packages


In [ ]:
# Not sure if this is needed.
# Reload functions during development
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline

from typing import Callable

# Scientific core
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.gridspec as gridspec
from matplotlib import pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

# Geospatial
import geopandas as gpd
import rioxarray # noqa: F401
import xarray as xr
import zarr # noqa: F401
from antimeridian import fix_multi_polygon
from ipyleaflet import basemaps
from odc.geo.xr import assign_crs
from odc.stac import load
from planetary_computer import sign_url
from pystac.client import Client
from shapely.geometry import box

# Machine learning
from scipy.ndimage import minimum_filter
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Local
from ldn.grids import get_gadm
from ldn.random_sampling import random_sampling
from ldn.typology import classes, colors, world_cover_map, cci_lc_map, io_map, lvl1
from ldn.utils import get_geomad_dem_indices
from notebooks.src.Compare_LULC_func import standardise_class

In [ ]:
# Parameters
datetime = '2020-06' # TODO: Just use 2020?
datetime_year = datetime.split("-")[0]

wgs84 = 'EPSG:4326' # WGS84
class_attr='lulc'
buffer_m  = 100
# Use Planetary Computer STAC API
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1/")


"""
# For Singapore:
country_of_interest = {"Singapore": "SGP"}
analysis_crs = 'EPSG:6933' # Equal Earth for global analysis
stac_geoparquet = "https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/ci_ls_geomad/0-0-2/ci_ls_geomad.parquet" # Non-Pacific
split_at_antimeridian = False # No chance that this country will intersect the antimeridian.
"""
# For Fiji:
country_of_interest = {"Fiji": "FJI"}
analysis_crs = 'EPSG:3832' # PDC Mercator for antimeridian avoiding
stac_geoparquet = "https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/ausp_ls_geomad/0-0-2/ausp_ls_geomad.parquet" # Pacific
split_at_antimeridian = True  # No chance that this country will intersect the prime meridian.

# WIP: For Fiji (and all Pacific countries to be safe against the antimeridian), I could do this whole process for east and west hemispheres and then merge the training data points at the end.

In [ ]:
# Make buffered country polygon to clip with.
country_wgs84_buffered = (
    get_gadm(countries=country_of_interest, overwrite=True)
    .to_crs(analysis_crs)
    .buffer(buffer_m)
    .to_crs(wgs84)
    # Fiji and Singapore are both a single multipolygon from GADM.
    .apply(fix_multi_polygon)
)
country_wgs84_buffered = gpd.GeoDataFrame(geometry=country_wgs84_buffered, crs=wgs84) # Turn the antimeridian fixed geometries back into a GeoDataFrame.
country_wgs84_buffered.explore()

# TODO: Why does this take 4 mins for Fiji? Is it the antimeridian fix?

In [ ]:
regions = [] # 2 for Pacific (East and West hemispheres), 1 for non-Pacific (no chance of crossing the antimeridian).

if split_at_antimeridian:
    print("2 zones are east and west hemispheres.")
    east = gpd.clip(country_wgs84_buffered, box(0, -90, 180, 90))
    west = gpd.clip(country_wgs84_buffered, box(-180, -90, 0, 90))
    regions = [east, west]
else:
    print("1 zone covering the whole globe and country. No splitting at the antimeridian needed.")
    regions = [country_wgs84_buffered]

print(f"Number of regions: {len(regions)}")
print("Region 0:")
regions[0].explore()

In [ ]:
assert len(regions[0].geometry) == 1

# This takes ~10 mins for region 0 of Fiji.
region_geomad_dem = []

for region in regions:
# for region in regions:
    region_geomad_dem.append(get_geomad_dem_indices(region, stac_geoparquet, datetime_year, catalog=catalog))

# Singapore and Fiji only have 2 GeoMAD tiles for 2020 as of 2026-03-05. GeoMAD will be fully processed in future.
# Elevation has the full extent of Fiji. GeoMedian/GeoMAD only have the processed tile extents.

In [ ]:
# # Write data as Zarr. Just read this to skip having to rerun the above code while developing the rest of the notebook.
# for i, ds in enumerate(region_geomad_dem):
#     ds.to_zarr(f"test_geomad_dem_{list(country_of_interest.keys())[0]}_region_{i}.zarr", mode="w")

region_geomad_dem2 = []
# Read for faster development.
for i, ds in enumerate(regions):
    region_geomad_dem2.append(xr.open_zarr(f"test_geomad_dem_{list(country_of_interest.keys())[0]}_region_{i}.zarr", consolidated=True))

print(region_geomad_dem2)

In [ ]:
# This cell is just to check the data visually.
for column in region_geomad_dem2[0].data_vars:
    # print(f"{column}: {country_geomad_dem[column].shape}, {country_geomad_dem[column].dtype}")
    print(f"Column: {column}, min: {region_geomad_dem2[0][column].min().values}, max: {region_geomad_dem2[0][column].max().values}, dtype: {region_geomad_dem2[0][column].dtype}")

# test_attr = "slope"
# test_attr = "elevation"
# test_attr = "aspect"
test_attr = "ndvi"
# test_attr = "red"
test_attr_colormap = {"elevation": "terrain", "aspect": "viridis", "slope": "plasma", "ndvi": "YlGn", "red": "Reds"}[test_attr]

region_geomad_dem2[0][test_attr].odc.explore(
    cmap=test_attr_colormap,
    vmin=region_geomad_dem2[0][test_attr].min().values,
    vmax=region_geomad_dem2[0][test_attr].max().values,
)

# For some reason for Fiji the data looks really low res, but it is actually 10m.
# Maybe .explore() downsamples because the extent is huge?
# Or maybe something is off with the Xarray Dataset coordinates.

In [ ]:
region_geomad_dem = region_geomad_dem2 # Overwrite loaded data with the Zarr version for the rest of the notebook.

## Load 3 LU/LC products for country.

In [ ]:
# Steps:
# For each region (1 for non-Pacific, 2 for Pacific):
# 1. Load the highest res product (there are 2 at 10m.)
# 2. Load the others to be like the 1st. Use nearest resampling to avoid creating new classes that may not exist in the original data.
# 3. Find agreement where 2 out of 3 of the products match exactly. CCI is probably going to agree the least due to its different resolution. Neighbours also agree.

# A generic function to load a product given the boundary
def load_lulc_data(region: gpd.GeoDataFrame, product: str, like: xr.Dataset | None) -> xr.Dataset:
    # Search across each bbox polygon (handles antimeridian splits)
    print("Search/Load bounds: ", list(region.total_bounds))
    assert datetime_year == "2020", "Year must be '2020'" # WC is 2020 and 2021. CCI is 1992 – 2020 (annual), and IO is 2017-2022 (annual). The only overlapping year across all three is 2020.
    lulc_items = catalog.search(
        collections=[product],
        bbox=list(region.total_bounds),
        datetime=datetime_year,
    )
    lulc_items = list(lulc_items.items())
    print(f"Found {len(lulc_items)} unique {product} LULC STAC items for this AOI")

    if like is not None:
        # Not first product — align to first product grid — like= handles extent correctly
        ds = load(
            lulc_items,
            like=like,
            chunks={},
            patch_url=sign_url,
            resampling="nearest",
        )
    else:
        # First product — load each bbox side separately to avoid antimeridian world-spanning extent
        ds = load(
            lulc_items,
            geopolygon=region,
            crs=wgs84,
            # resolution=resolution, # Load native resolution. WGS84 is in degrees so using resolution in metres breaks loading.
            chunks={},
            patch_url=sign_url,
            resampling="nearest",
        )

    print(f"Product {product} has variables: {ds.data_vars}")
    return ds

In [ ]:
# This cell takes 25 mins for Fiji (2 regions, 3 products). (Before I added quality masking).

LULC_PRODUCTS = [
    {
        "product":     "esa-worldcover", # 10m
        "native_band": "map",
        "output_band": "esa_wc",
        "class_map":   world_cover_map,
        # Retain pixels with at least 1 valid Sentinel-1/2 observation in each of the 3 seasons.
        # TODO: Relax this to 2/3. It filters a lot of Fiji.
        "quality_fn":  lambda ds: (ds["input_quality.1"] > 0) & (ds["input_quality.2"] > 0) & (ds["input_quality.3"] > 0),
        "quality_bands": ["input_quality.1", "input_quality.2", "input_quality.3"],
    },
    {
        "product":     "esa-cci-lc", # 300m
        "native_band": "lccs_class",
        "output_band": "esa_cci",
        "class_map":   cci_lc_map,
        # Retain pixels that were processed, had a stable class across the time series, and had at least 3 observations.
        # TODO: Check how much this filters.
        "quality_fn":  lambda ds: (ds["processed_flag"] == 1) & (ds["change_count"] == 0) & (ds["observation_count"] >= 3),
        "quality_bands": ["processed_flag", "change_count", "observation_count"],
    },
    {
        "product":     "io-lulc-annual-v02", # 10m
        "native_band": "data",
        "output_band": "io",
        "class_map":   io_map,
        # No quality band available
        "quality_fn":  None,
        "quality_bands": [],
    },
]


def load_and_prepare(region: gpd.GeoDataFrame, product: str, native_band: str, output_band: str, class_map: dict, quality_fn: Callable[[xr.Dataset], xr.DataArray] | None, quality_bands: list[str], like: xr.Dataset | None) -> xr.Dataset:
    ds = load_lulc_data(region, product, like=like)
    ds[output_band] = ds[native_band]

    # Keep quality bands until after masking
    bands_to_keep = {output_band} | set(quality_bands)
    ds = ds.drop_vars([v for v in ds.data_vars if v not in bands_to_keep])
    ds = ds.isel(time=0)

    # Set spatial dims and CRS while still lazy
    ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
    ds = ds.rio.write_crs(wgs84, inplace=True)

    # Clip and quality mask before .load() to avoid materialising full tiles
    print("Before clipping")
    print(f"Dataset CRS: {ds.odc.crs.epsg}")
    print(f"Dataset bounds: {ds.odc.geobox.boundingbox}")
    print(f"Region CRS: {region.crs.to_epsg()}")
    print(f"Region bounds: {region.total_bounds}")
    # Precise clip to actual region geometry
    ds = ds.rio.clip(region.geometry.values, crs=wgs84, from_disk=True, all_touched=True)
    print("After clipping")
    print(f"Dataset bounds: {ds.odc.geobox.boundingbox}")
    xmin, ymin, xmax, ymax = ds.odc.geobox.boundingbox
    assert xmin >= -180.0 and xmax <= 180.0, f"Dataset bounds exceed world bounds after clipping: {ds.odc.geobox.boundingbox}"

    # Don't do quality masking while debugging antimeridian stuff.
    # TODO: Enable quality masking.
    # # Quality mask
    # if quality_fn is not None:
    #     ds[output_band] = ds[output_band].where(quality_fn(ds))

    # Drop quality bands then load only the clipped, masked result
    ds = ds.drop_vars([v for v in ds.data_vars if v != output_band])
    ds = ds.load()

    ds[output_band] = standardise_class(ds[output_band], class_map)
    return ds

region_product = []

# for i, region in enumerate(regions[:1]): # debugging
for i, region in enumerate(regions):
    # Load worldcover first (highest native res), then align others to it
    wc_10m,  cci_10m,  io_10m  = None, None, None

    # for j, p in enumerate(LULC_PRODUCTS[:1]): # debugging
    for j, p in enumerate(LULC_PRODUCTS):
        like = wc_10m if j > 0 else None
        ds = load_and_prepare(region, p["product"], p["native_band"], p["output_band"], p["class_map"], p["quality_fn"], p["quality_bands"], like)
        if j == 0:
            wc_10m  = ds
        elif j == 1:
            cci_10m = ds
        else:
            io_10m  = ds

    region_product.append((region, wc_10m, cci_10m, io_10m)) # region isn't really needed in tuple.

In [ ]:
# Write products data as Zarr. Just read this to skip having to rerun the above code while developing the rest of the notebook.
for i, ds in enumerate(region_product):
    _, wc_10m, cci_10m, io_10m = ds
    wc_10m.to_zarr(f"test_product_{list(country_of_interest.keys())[0]}_region_{i}_wc.zarr", mode="w")
    cci_10m.to_zarr(f"test_product_{list(country_of_interest.keys())[0]}_region_{i}_cci.zarr", mode="w")
    io_10m.to_zarr(f"test_product_{list(country_of_interest.keys())[0]}_region_{i}_io.zarr", mode="w")

# region_product[0][1].to_zarr(f"test_product_{list(country_of_interest.keys())[0]}_region_{i}_wc_3.zarr", mode="w")

# region_product2 = []
# # Read for faster development.
# for i, ds in enumerate(regions):
#     wc = xr.open_zarr(f"test_product_{list(country_of_interest.keys())[0]}_region_{i}_wc.zarr", consolidated=True)
#     cci = xr.open_zarr(f"test_product_{list(country_of_interest.keys())[0]}_region_{i}_cci.zarr", consolidated=True)
#     io = xr.open_zarr(f"test_product_{list(country_of_interest.keys())[0]}_region_{i}_io.zarr", consolidated=True)
#     region_product2.append((ds, wc, cci, io))

# print(region_product2)
# region_product = region_product2

In [ ]:
# Print class distributions before agreement analysis.
for i, rp in enumerate(region_product):
    print (f"Class distribution for region {i}")
    wc_10m, cci_10m, io_10m = rp[1], rp[2], rp[3]
    
    for name, ds, band in [
        ("wc", wc_10m, 'esa_wc'),
        ("cci",        cci_10m, 'esa_cci'),
        ("io",         io_10m,  'io'),
    ]:
        vals, counts = np.unique(ds[band].values, return_counts=True)
        print(name)
        print(pd.DataFrame({"class": vals, "count": counts}))

    # CCI has no Other (7) class in Singapore or Fiji.
    # Fiji has a lot of nodata (0) because of vast oceas between islands. This is expected.


# Create layer where all 2 out of 3 land cover products agree.
Each pixel and its 8 neighbours must have agreement for 2 out of the 3 products.

In [ ]:
region_agreeing = []

for i, rp in enumerate(region_product):
    print(f"Finding agreement for region {i}")
    wc_10m, cci_10m, io_10m = rp[1], rp[2], rp[3]

    # Count pairwise agreements at each pixel (0, 1, 2, or 3 pairs agree)
    wc  = wc_10m['esa_wc']
    # Snap to WC grid. Needed. Maybe caused by clipping?
    cci = cci_10m['esa_cci'].reindex_like(wc, method="nearest", fill_value=0)
    io  = io_10m['io'].reindex_like(wc, method="nearest", fill_value=0)

    # Valid pixels — all three products have data for this region.
    valid = (wc > 0) & (cci > 0) & (io > 0)

    pair_agree_count = (
        (wc == cci).astype("int8") +
        (cci == io).astype("int8") +
        (wc == io).astype("int8")
    )

    # At least 2 of 3 products agree — take the majority class
    majority_class = xr.where(wc == cci, wc, io)  # if wc==cci, that's the majority; else io agrees with one of them
    two_of_three = (pair_agree_count >= 2) & valid

    # All 8 neighbours must also have 2/3 agreement
    neighbour_mask = xr.DataArray(
        minimum_filter(two_of_three.values.astype("float32"), size=3, mode="constant", cval=0) == 1,
        coords=two_of_three.coords, dims=two_of_three.dims,
    )

    agreed_class = majority_class.where(neighbour_mask & two_of_three, other=0).rename(class_attr)

    # Diagnostics
    label_map = {c["value"]: c["label"] for c in lvl1.values()}
    _classes, counts = np.unique(agreed_class.values, return_counts=True)
    counts_df = pd.DataFrame({"class": _classes, "pixel_count": counts})
    counts_df["label"] = counts_df["class"].map(label_map)
    print(counts_df)

    # Write agreeing pixels as tiff.
    agreed_class.odc.write_cog(f'lulc_agree_{list(country_of_interest.keys())[0]}_region{i}.tif', overwrite=True)

    region_agreeing.append(agreed_class)

# Visualise the agreeing classes from 3 products

In [ ]:
# for i, ra in enumerate(region_agreeing):
#     with plt.ioff():  # prevents double-rendering in some notebook backends
#         fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        
#         vals, counts = np.unique(ra.values, return_counts=True)
#         print(f"Region {i}:")
#         print(pd.DataFrame({"class": vals, "count": counts}))

#         cmap = ListedColormap([colors[k] for k in vals])
#         norm = BoundaryNorm(list(vals) + [np.max(vals) + 1], cmap.N)
#         ra.plot.imshow(ax=ax, cmap=cmap, norm=norm, add_labels=True, add_colorbar=False, interpolation='none')
#         patches_list = [Patch(facecolor=colors[k]) for k in colors]
#         ax.legend(patches_list, list(classes.keys()), loc='upper center', ncol=4, bbox_to_anchor=(0.5, -0.1))
#         ax.set_title(f"Region {i} — agreed LULC class")

#         display(fig)
#         plt.close(fig)

## Generate random training samples
We generate some randomly distributed samples for each class from the clipped classification map using the `random_sampling` function. This function takes in a few parameters:  
* `da`: a classified map in the format of 2-dimensional xarray.DataArray
* `n`: total number of points to sample.
* `min_sample_n`: Minimum number of samples to generate per class if proportional number is smaller than this. **Note that the resultant number of samples may be higher than the set `n` due to setting of this minimum number of samples.** 
* `sampling`: the sampling strategy, e.g. 'stratified_random' where each class has a number of points proportional to its relative area, 'equal_stratified_random' where each class has the same number of points, or 'manual' which allows you to define number of samples for each class.
* `out_fname`: a filepath name for the function to export a shapefile/geojson of the sampling points into a file. You can set this to `None` if you don't need to output the file.
* `class_attr`: This is the column name of output dataframe that contains the integer class values on the classified map. 
* `drop_value`: pixel value on the classification map to be excluded from sampling.

The output of the function is a geopandas dataframe of randomly distributed points containing a column `class_attr` identifying class values. 

Here we extract around 1000 training points in total and export the points in a geojson file for use in the rest of workflow. Here we use the stratified sampling method by setting 'equal_stratified_random', but also set the minimum number of samples as 75 to avoid missing samples for some minor classes. 

As mentioned earlier we don't want the abandoned classes to be included in the samples we set drop_value as 0 before implementing the function:

In [ ]:
# TODO: I am up to here for Fiji.
# TODO: Validate when to merge regions. Not doing it yet.

In [ ]:
n=2100
min_sample_n=300
drop_value=0

# I clip the products in load_and_prepare so they should be real areas of the regions.
# Proportional sampling based on area of the regions to ensure representative samples from all regions.
region_areas = [r.to_crs(analysis_crs).area for r in regions]
total_area = sum(region_areas)
region_proportional_areas = [r/total_area for r in region_areas]

region_n = [int(n * prop) for prop in region_proportional_areas]
region_min_sample_n = [int(min_sample_n * prop) for prop in region_proportional_areas]

region_samples = []
for i, ra in enumerate(region_agreeing):
    region_samples.append(
        random_sampling(
            da=ra,
            n=region_n[i],
            sampling='stratified_random',
            min_sample_n=region_min_sample_n[i],
            out_fname=f"region_{i}_samples.geojson", # None
            class_attr=class_attr,
            drop_value=drop_value,
        )
    )

In [ ]:
region_samples[0].head()

## Visualise the training data by class

In [ ]:
region_samples[0].explore(
    column=class_attr,
    categorical=True,
    categories=(present_classes := sorted(region_samples[0][class_attr].unique())), # Walrus operator to assign and pass the value in one step.
    cmap=[colors[c] for c in present_classes],
    legend=True,
    style_kwds={"radius": 6, "fillOpacity": 0.8, "weight": 0.5}
)

## Extract Geomedian/GeoMAD values at training data point locations.

For now we run and load individual tiles and mosaic them because we have not run all tiles yet.

Once we have finalised the GeoMAD data, we will set up a GeoParquet STAC index for it. Then querying it by bbox will be simple and replace the more complicated process implemented here.

In [ ]:
for i, rgd in enumerate(region_geomad_dem):
    print(f"Region {i} GeoMAD DEM bands:")
    band_names_with_indices = [v for v in rgd.data_vars]
    print(band_names_with_indices)

    # Extract Geomedian and GeoMAD elevation, indices values at each sample point location

    # Reproject sample points to match native CRS of geomad before sampling.
    gpd_random_samples_analysis = region_samples[i].to_crs(analysis_crs)

    # Build point-wise x/y arrays with the SAME 'points' index
    points_idx = np.arange(len(gpd_random_samples_analysis))
    xs = xr.DataArray(gpd_random_samples_analysis.geometry.x.values, dims="points", coords={"points": points_idx})
    ys = xr.DataArray(gpd_random_samples_analysis.geometry.y.values, dims="points", coords={"points": points_idx})
    # Point-wise nearest sampling (one sampled pixel per input point)
    sampled = rgd[band_names_with_indices].sel(x=xs, y=ys, method="nearest")

    # Convert to dataframe indexed by points; keep exactly one row per point
    sampled_df = sampled.to_dataframe().reset_index()
    sampled_df = sampled_df.sort_values("points").reset_index(drop=True)

    # Merge back to training points (original CRS)
    training_data = region_samples[i].reset_index(drop=True).copy()
    training_data = pd.concat([training_data, sampled_df[band_names_with_indices]], axis=1)

    print(f"Training data shape: {training_data.shape}")
    print("Unique values per band:")
    print(training_data[band_names_with_indices].nunique())
    training_data.head()

    # Update region_samples.
    region_samples[i] = training_data

## Visualise GeoMedian red values per sample point

In [ ]:
# Visualise red values using the actual data range
region_samples[0].explore(
    column="red",
    cmap="Reds",
    k=7,
    scheme="natural_breaks",
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

## Explore elevation data on training data points

In [ ]:
region_samples[0].explore(
    column="elevation",
    cmap="Blues",
    scheme="natural_breaks",
    k=7,
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

## Explore NDVI data on training data points

In [ ]:
region_samples[0].explore(
    column="ndvi",
    cmap="RdYlGn",
    scheme="quantiles",
    k=7,
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

# Section 2: Filter training data by K-Means clustering
Goal is to reduce confusion. Make training point classes clean/accurate/spectrally separable.

In [ ]:
for i, training_data in enumerate(region_samples):
    print(f"Region {i}:")

    # 1. Prep feature matrix ────────────────────────────────────────────────────
    exclude_cols = ['lulc', 'geometry']
    feature_cols = [c for c in training_data.columns if c not in exclude_cols]

    training_data['outlier'] = False
    training_data.loc[training_data[feature_cols].isna().any(axis=1), 'outlier'] = True

    nan_rows = training_data['outlier'].sum()
    print(f"Rows with NaNs (auto-flagged): {nan_rows}")

    valid_mask = ~training_data[feature_cols].isna().any(axis=1)
    valid_idx  = training_data.index[valid_mask]

    X = training_data.loc[valid_idx, feature_cols].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 2. Visualisation helpers ──────────────────────────────────────────────────
    CLUSTER_CMAP = 'tab10'

    def plot_pca(ax, Xc, labels, centers, outlier_mask, class_name, best_score):
        """PCA scatter — inliers coloured by cluster, outliers as red crosses."""
        pca      = PCA(n_components=2)
        X_2d     = pca.fit_transform(Xc)
        c_2d     = pca.transform(centers)
        var      = pca.explained_variance_ratio_
        k        = len(np.unique(labels))
        inliers  = ~outlier_mask

        ax.scatter(
            X_2d[inliers, 0], X_2d[inliers, 1],
            c=labels[inliers], cmap=CLUSTER_CMAP, vmin=0, vmax=max(k - 1, 1),
            alpha=0.55, s=25, linewidths=0, label='inlier'
        )
        if outlier_mask.any():
            ax.scatter(
                X_2d[outlier_mask, 0], X_2d[outlier_mask, 1],
                c='red', marker='x', s=50, linewidths=1.2, label='outlier'
            )
        ax.scatter(
            c_2d[:, 0], c_2d[:, 1],
            c='black', marker='*', s=180, zorder=5, label='centroid'
        )
        ax.set(
            xlabel=f'PC1 ({var[0]:.1%})',
            ylabel=f'PC2 ({var[1]:.1%})',
            title=f'PCA  |  k={k}  sil={best_score:.3f}'
        )
        ax.legend(fontsize=7, markerscale=0.9)


    def plot_heatmap(ax, centers, feature_cols, outlier_count, n):
        """Centroid heatmap — rows = clusters, cols = features."""
        im = ax.imshow(centers, aspect='auto', cmap='RdBu_r')
        ax.set_xticks(range(len(feature_cols)))
        ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(len(centers)))
        ax.set_yticklabels([f'C{i}' for i in range(len(centers))], fontsize=8)
        plt.colorbar(im, ax=ax, label='Scaled value', pad=0.02)

        # Annotate each cell with the scaled value
        for r in range(centers.shape[0]):
            for c in range(centers.shape[1]):
                ax.text(c, r, f'{centers[r, c]:.1f}',
                        ha='center', va='center', fontsize=6,
                        color='white' if abs(centers[r, c]) > 1.2 else 'black')

        ax.set_title(f'Centroids  |  outliers={outlier_count} ({100*outlier_count/n:.1f}%)')


    # 3. Per-class clustering + outlier flagging + visualisation ────────────────
    threshold = 2.0   # ← tune: lower = more aggressive flagging

    _classes = training_data.loc[valid_idx, 'lulc'].unique()

    for cls in _classes:
        mask = (training_data.loc[valid_idx, 'lulc'] == cls).values
        idx  = valid_idx[mask]
        Xc   = X_scaled[mask]
        n    = len(Xc)

        if n < 6:
            continue
        k_max = min(5, n // 5)
        if k_max < 2:
            continue

        # Optimal k via silhouette ──────────────────────────────────────────
        best_k, best_score = 2, -1
        for k in range(2, k_max + 1):
            km     = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = km.fit_predict(Xc)
            score  = silhouette_score(Xc, labels)
            if score > best_score:
                best_k, best_score = k, score

        km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(Xc)
        labels   = km_final.labels_
        centers  = km_final.cluster_centers_

        # Outlier flagging ──────────────────────────────────────────────────
        outlier_mask = np.zeros(n, dtype=bool)
        for cluster_id in range(best_k):
            in_cluster = labels == cluster_id
            pts        = Xc[in_cluster]
            dists      = cdist(pts, centers[cluster_id].reshape(1, -1)).flatten()
            median_d   = np.median(dists)
            if median_d == 0:
                continue
            flagged = dists > threshold * median_d
            outlier_mask[np.where(in_cluster)[0][flagged]] = True

        training_data.loc[idx[outlier_mask], 'outlier'] = True

        print(f"  {str(cls):30s} | n={n:4d} | k={best_k} | sil={best_score:.3f} "
            f"| outliers={outlier_mask.sum():4d} ({100*outlier_mask.mean():.1f}%)")

        # Figure: PCA (left) + heatmap (right) ─────────────────────────────
        fig = plt.figure(figsize=(14, 4.5))
        fig.suptitle(f'Class: {cls}  (n={n})', fontsize=12, fontweight='bold', y=1.01)

        gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1.6], figure=fig)
        ax_pca  = fig.add_subplot(gs[0])
        ax_heat = fig.add_subplot(gs[1])

        plot_pca(ax_pca, Xc, labels, centers, outlier_mask, cls, best_score)
        plot_heatmap(ax_heat, centers, feature_cols, outlier_mask.sum(), n)

        plt.tight_layout()
        plt.show()

    # 4. Summary ────────────────────────────────────────────────────────────────
    print(f"\nTotal points : {len(training_data):,}")
    print(f"Clean        : {(~training_data['outlier']).sum():,}")
    print(f"Outliers     : {training_data['outlier'].sum():,}")
    print("Outliers per class:")
    print(training_data.groupby('lulc')['outlier'].mean().mul(100).round(1).astype(str) + '%')
    training_data = training_data[~training_data['outlier']]

    # Update region_samples.
    region_samples[i] = training_data

In [ ]:
for i, training_data in enumerate(region_samples):
    print(f"Region {i}:")
    training_data.drop(columns=["time", 'spatial_ref', 'count'], inplace=True)

    out_fname = f'training_data_region{i}'

    # Write the final training data
    training_data.to_file(f"{out_fname}.geojson", driver="GeoJSON", index=False)
    training_data.to_csv(f"{out_fname}.csv", index=False)

    print(f"Saved training data as geojson and CSV to {out_fname}")

# Section 3: Train and Predict

In [ ]:
# TODO: Should each region have its own model? Or make one model for all regions and then predict per region to avoid data search/load issues?

# Find and load training data for each region

for i, geomad_dem in enumerate(region_geomad_dem):
    training_data_file = f"training_data_region{i}.csv"
    training_data = pd.read_csv(training_data_file)
    print(training_data.columns)

    training_data.drop(columns=['outlier'], inplace=True)

    # Split 70/30 into train/test. Splits the classes into train/test in a representative way.
    train_gdf, test_gdf = train_test_split(training_data, test_size=0.3, stratify=training_data[class_attr], random_state=42)

    print(f"Training set class distribution:\n{train_gdf[class_attr].value_counts()}")
    print(f"Test set class distribution:\n{test_gdf[class_attr].value_counts()}")
    print(train_gdf)

    ## Create a classifier and fit a model

    _classes = train_gdf[class_attr].values.astype(int)
    print(f"Classes: {np.unique(_classes)}")

    # Observations — drop class and any remaining non-numeric columns
    feature_cols = [c for c in train_gdf.columns
                    if c != class_attr
                    and pd.api.types.is_numeric_dtype(train_gdf[c])]
    observations = train_gdf[feature_cols].values

    # Create and fit
    classifier = RandomForestClassifier(class_weight='balanced')
    model = classifier.fit(observations, _classes)
    # Define features and target

    feature_cols = [c for c in train_gdf.columns if c != class_attr]

    X_train = train_gdf[feature_cols].values
    y_train = train_gdf[class_attr].values
    X_test = test_gdf[feature_cols].values
    y_test = test_gdf[class_attr].values

    classifier = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
    model = classifier.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Feature importance — drop noisy features
    importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print("Feature importances:")
    print(importances)
    # Feature importance is probably the most useful next step — it'll tell you which bands are actually helping and which are adding noise.

    present = np.unique(np.concatenate([y_test, y_pred]))
    target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v in present]

    print(classification_report(y_test, y_pred, target_names=target_names, labels=present))

    stack = np.stack([geomad_dem[f].values.flatten() for f in feature_cols], axis=1)
    stack = np.stack([geomad_dem[f].values.flatten() for f in feature_cols], axis=1)
    stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    predictions = model.predict(stack)

    # Reshape back to raster
    prediction_map = predictions.reshape(geomad_dem[feature_cols[0]].shape)

    # Wrap in DataArray
    predicted_da = xr.DataArray(
        prediction_map,
        coords={"y": geomad_dem.y, "x": geomad_dem.x},
        dims=["y", "x"],
        name="lulc",
    )

    ## Visualise our results

    predicted_da = assign_crs(predicted_da, crs=analysis_crs)

    # Can't use this because there is no Other (7) class in prediction.
    # data_classes = sorted(colors.keys())
    # print(f"All classes: {data_classes}")
    data_classes = [0, 1, 2, 3, 4, 5, 6]
    cmap = ListedColormap([colors[c] for c in data_classes])

    predicted_da.odc.explore(
        classes=data_classes,
        cmap=cmap,
        legend=True,
        tiles=basemaps.Esri.WorldImagery
    )

    # Aim for >80% accuracy. Don't just look at the confusion matrix, also look at the output map.

    # Use a product for validation.
    # One validation method for tuning and another for final measure.

    # target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v != 0]
    target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v != 0 and v != 7] # Use this while there is no Other (7) class in the data.

    # Classification report
    print(classification_report(y_test, y_pred, target_names=target_names))
    print(classification_report(y_test, y_pred, target_names=target_names))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(xticks_rotation=45, cmap="Blues")

    predicted_da.odc.write_cog(f"predicted_lulc_region{i}.tif", overwrite=True)